# Latent Probe Diagnostic

Tests how much task-relevant information lives in the world-model latents.
For every step in every eval episode we extract two latent trajectories:

1. **Posterior** — drive encoder + RSSM with the real obs stream. Latents reflect
   what the encoder + posterior net can extract directly from observations.
2. **Prior** — burn the posterior in for ``burn_in`` steps, then continue
   open-loop. Latents reflect what the dynamics alone can predict from the
   anchor state.

For each variant we train two probes — a **linear** ridge and a small **MLP** —
from latent features (``h``, ``s``, or ``[h, s]``) to three ground-truth targets:
``object_xy``, ``ee_xy``, and per-joint ``q_i``. Train/test split is at the
**episode** level so the probe can't cheat by interpolating consecutive frames.

Episode loading + latent extraction live in `chuck_dreamer.eval` — only the
probe code stays in the notebook.

In [ ]:
# Parameters — papermill injects overrides after this cell.
checkpoint_path = "checkpoints/default/final.safetensors"
data_path       = "data/eval"
data_format     = "rerun"
num_episodes    = 40
burn_in         = 5
horizon         = 15
test_frac       = 0.25
mlp_hidden      = (64, 64)
mlp_epochs      = 200
mlp_lr          = 1e-3
mlp_batch       = 512
ridge_alpha     = 1e-3
seed            = 0

## Setup

In [ ]:
import json

import matplotlib.pyplot as plt
import mlx.core as mx
import mlx.nn as mlxnn
import mlx.optimizers as mlxopt
import numpy as np

from chuck_dreamer.eval import load_checkpoint, load_eval_episodes, run_split_rollout

rng = np.random.default_rng(seed)
mx.random.seed(seed)

print(f"checkpoint:   {checkpoint_path}")
print(f"data:         {data_path} ({data_format})")
print(f"num_episodes: {num_episodes}")
print(f"burn_in:      {burn_in}, horizon: {horizon}")
print(f"test_frac:    {test_frac}")

## Load model and evaluation episodes

We need both the processed episode (encoder input) and the raw episode
(ground truths the modal processor drops: `object_xy`, `ee_pos`,
`joint_qpos`). `EvalEpisode` carries both.

In [ ]:
ckpt = load_checkpoint(checkpoint_path)
obs_mode = ckpt.env.obs_mode
stoch_dim = ckpt.model.rssm.stoch_dim
deter_dim = ckpt.model.rssm.deter_dim
print(f"obs_mode={obs_mode}  act_mode={ckpt.env.act_mode}")
print(f"RSSM dims: stoch={stoch_dim}, deter={deter_dim}")

sources = [{"path": data_path, "format": data_format, "num_episodes": num_episodes}]
min_len = burn_in + horizon

per_episode: list[dict] = []
for episode in load_eval_episodes(ckpt.config, sources=sources,
                                  min_len=min_len, max_episodes=num_episodes):
  T = episode.num_actions
  # Full posterior across the whole episode.
  post = run_split_rollout(
    ckpt.model, episode,
    burn_in=0, horizon=T, obs_mode=obs_mode,
    decode=False, predict_reward=False,
  )
  # Posterior burn-in + prior tail, sharing the same anchor.
  prior_tail = run_split_rollout(
    ckpt.model, episode,
    burn_in=burn_in, horizon=T - burn_in, obs_mode=obs_mode,
    decode=False, predict_reward=False,
  )
  if post is None or prior_tail is None:
    continue
  # Stitch a length-T prior trajectory: posterior latents during burn-in,
  # then the prior tail (which already starts from the burn-in anchor).
  h_post = post.h_posterior
  s_post = post.s_posterior
  h_prior = np.concatenate([h_post[:burn_in], prior_tail.h_prior], axis=0)
  s_prior = np.concatenate([s_post[:burn_in], prior_tail.s_prior], axis=0)

  raw = episode.raw
  object_xy = np.asarray(raw["object_xy"], dtype=np.float32)[:T]
  ee_pos    = np.asarray(raw["ee_pos"],    dtype=np.float32)[:T]
  joints    = np.asarray(raw["joint_qpos"], dtype=np.float32)[:T]

  per_episode.append({
    "T":         T,
    "h_post":    h_post[:T],
    "s_post":    s_post[:T],
    "h_prior":   h_prior[:T],
    "s_prior":   s_prior[:T],
    "object_xy": object_xy,
    "ee_xy":     ee_pos[:, :2],
    "joints":    joints,
  })
if not per_episode:
  raise RuntimeError("No usable eval episodes — check data_path / min_len.")
n_joints = per_episode[0]["joints"].shape[-1]
print(f"Extracted latents + targets for {len(per_episode)} episodes; n_joints={n_joints}")

## Pool & split

Episode-level split so the held-out set never shares timesteps with the
training set. Step indices are tracked so we can mask the test set to the
open-loop window when evaluating prior probes.

In [ ]:
ep_indices = np.arange(len(per_episode))
rng.shuffle(ep_indices)
n_test = max(1, int(round(test_frac * len(ep_indices))))
test_eps  = set(ep_indices[:n_test].tolist())
train_eps = set(ep_indices[n_test:].tolist())
print(f"Train: {len(train_eps)} eps  |  Test: {len(test_eps)} eps")

def _pool(eps, key, source):
  """Concatenate latent features. key ∈ {h,s,hs}; source ∈ {post,prior}."""
  xs, ts = [], []
  for i in eps:
    rec = per_episode[i]
    h = rec[f"h_{source}"]; s = rec[f"s_{source}"]
    if key == "h":   xs.append(h)
    elif key == "s": xs.append(s)
    else:            xs.append(np.concatenate([h, s], axis=-1))
    ts.append(np.arange(rec["T"], dtype=np.int32))
  return np.concatenate(xs, axis=0), np.concatenate(ts, axis=0)

def _pool_target(eps, target_key, col=None):
  ys = []
  for i in eps:
    y = per_episode[i][target_key]
    if col is not None:
      y = y[:, col:col + 1]
    ys.append(y)
  return np.concatenate(ys, axis=0)

## Probes

* **Linear** — closed-form ridge in standardized space.
* **MLP** — small MLX MLP, plain Adam + MSE.

Both standardize x and y on train statistics, report R² / RMSE in the
original target space.

In [ ]:
def _standardize(x, y):
  xm = x.mean(0, keepdims=True); xs = x.std(0, keepdims=True) + 1e-6
  ym = y.mean(0, keepdims=True); ys = y.std(0, keepdims=True) + 1e-6
  return xm, xs, ym, ys

def r2_score(y_true, y_pred):
  ss_res = float(((y_true - y_pred) ** 2).sum())
  ss_tot = float(((y_true - y_true.mean(0, keepdims=True)) ** 2).sum())
  return 1.0 - ss_res / max(ss_tot, 1e-12)

def rmse(y_true, y_pred):
  return float(np.sqrt(((y_true - y_pred) ** 2).mean()))

def fit_linear_probe(Xtr, ytr, Xte, yte, alpha):
  xm, xs, ym, ys = _standardize(Xtr, ytr)
  Xs_n = (Xtr - xm) / xs; Ys_n = (ytr - ym) / ys
  Xb = np.concatenate([Xs_n, np.ones((Xs_n.shape[0], 1), dtype=Xs_n.dtype)], axis=1)
  d = Xb.shape[1]
  A = Xb.T @ Xb + alpha * np.eye(d, dtype=Xb.dtype)
  A[-1, -1] -= alpha
  W = np.linalg.solve(A, Xb.T @ Ys_n)
  def apply(X):
    Xn = (X - xm) / xs
    Xb = np.concatenate([Xn, np.ones((Xn.shape[0], 1), dtype=Xn.dtype)], axis=1)
    return (Xb @ W) * ys + ym
  return apply(Xtr), apply(Xte), apply

class _ProbeMLP(mlxnn.Module):
  def __init__(self, in_dim, hidden, out_dim):
    super().__init__()
    layers: list = []
    prev = in_dim
    for h in hidden:
      layers.append(mlxnn.Linear(prev, h)); layers.append(mlxnn.ReLU()); prev = h
    layers.append(mlxnn.Linear(prev, out_dim))
    self.net = mlxnn.Sequential(*layers)
  def __call__(self, x):
    return self.net(x)

def fit_mlp_probe(Xtr, ytr, Xte, yte, *, hidden, epochs, lr, batch):
  xm, xs, ym, ys = _standardize(Xtr, ytr)
  Xs_n = (Xtr - xm) / xs; Ys_n = (ytr - ym) / ys
  net = _ProbeMLP(Xs_n.shape[1], hidden, Ys_n.shape[1])
  opt = mlxopt.Adam(learning_rate=lr)
  def loss_fn(model, x, y):
    return ((model(x) - y) ** 2).mean()
  loss_and_grad = mlxnn.value_and_grad(net, loss_fn)
  Xmx = mx.array(Xs_n.astype(np.float32)); Ymx = mx.array(Ys_n.astype(np.float32))
  n = Xmx.shape[0]
  for _ in range(epochs):
    perm = np.random.permutation(n)
    for start in range(0, n, batch):
      idx = mx.array(perm[start:start + batch])
      _, grads = loss_and_grad(net, Xmx[idx], Ymx[idx])
      opt.update(net, grads)
      mx.eval(net.parameters(), opt.state)
  def apply(X):
    Xn = (X - xm) / xs
    return np.asarray(net(mx.array(Xn.astype(np.float32)))) * ys + ym
  return apply(Xtr), apply(Xte), apply

## Sweep

For every (source × latent × probe × target) combination, fit on the train
pool and report R² / RMSE on the held-out pool. For ``source=prior`` we mask
the test set to the open-loop window (`t >= burn_in`) — that's where prior
and posterior actually diverge.

In [ ]:
LATENT_KEYS = ("h", "s", "hs")
SOURCES     = ("post", "prior")
PROBES      = ("linear", "mlp")

def _open_loop_mask(t_idx, source):
  if source == "post":
    return np.ones_like(t_idx, dtype=bool)
  return t_idx >= burn_in

def _run_one(target_name, target_key, col, eps_tr, eps_te):
  rows = []
  y_tr = _pool_target(eps_tr, target_key, col=col)
  y_te = _pool_target(eps_te, target_key, col=col)
  for source in SOURCES:
    for key in LATENT_KEYS:
      X_tr, _    = _pool(eps_tr, key, source)
      X_te, t_te = _pool(eps_te, key, source)
      mask_te = _open_loop_mask(t_te, source)
      _, lin_pred_te, _ = fit_linear_probe(X_tr, y_tr, X_te, y_te, alpha=ridge_alpha)
      _, mlp_pred_te, _ = fit_mlp_probe(X_tr, y_tr, X_te, y_te,
                                        hidden=mlp_hidden, epochs=mlp_epochs,
                                        lr=mlp_lr, batch=mlp_batch)
      for probe_name, pred in (("linear", lin_pred_te), ("mlp", mlp_pred_te)):
        rows.append({
          "target": target_name, "source": source, "latent": key, "probe": probe_name,
          "r2":     r2_score(y_te[mask_te], pred[mask_te]),
          "rmse":   rmse(y_te[mask_te], pred[mask_te]),
          "n_eval": int(mask_te.sum()),
        })
  return rows

results_rows: list[dict] = []
eps_tr = sorted(train_eps); eps_te = sorted(test_eps)
print("Probing object_xy ...")
results_rows += _run_one("object_xy", "object_xy", None, eps_tr, eps_te)
print("Probing ee_xy ...")
results_rows += _run_one("ee_xy", "ee_xy", None, eps_tr, eps_te)
print(f"Probing {n_joints} joints ...")
for j in range(n_joints):
  results_rows += _run_one(f"q_{j}", "joints", j, eps_tr, eps_te)
print(f"Total probe results: {len(results_rows)}")

## Task-relevant targets

`object_xy` is hardest (contact dynamics). `ee_xy` is mid (close to a
function of joint angles). Joints are easiest (action-conditioned).
The gap between posterior and open-loop prior on `object_xy` is the
metric that matters most for imagination-based actor training.

In [ ]:
def _by(rows, **filters):
  return [r for r in rows if all(r[k] == v for k, v in filters.items())]

def _plot_grouped_r2(rows, title):
  variants = [("post", "linear"), ("post", "mlp"),
              ("prior", "linear"), ("prior", "mlp")]
  colors  = {"post": "C0", "prior": "C3"}
  hatches = {"linear": "", "mlp": "//"}
  width   = 0.18
  xs      = np.arange(len(LATENT_KEYS))
  fig, ax = plt.subplots(figsize=(8, 4))
  for k, (src, probe) in enumerate(variants):
    vals = []
    for key in LATENT_KEYS:
      hit = _by(rows, source=src, latent=key, probe=probe)
      vals.append(hit[0]["r2"] if hit else np.nan)
    offset = (k - (len(variants) - 1) / 2) * width
    ax.bar(xs + offset, vals, width, color=colors[src], edgecolor="black",
           hatch=hatches[probe], label=f"{src} / {probe}")
  ax.set_xticks(xs); ax.set_xticklabels(LATENT_KEYS)
  ax.set_xlabel("latent input"); ax.set_ylabel("R² (test)")
  ax.set_ylim(min(-0.05, ax.get_ylim()[0]), 1.05)
  ax.axhline(0, color="k", linewidth=0.5, alpha=0.4)
  ax.set_title(title); ax.legend(ncol=2, fontsize=8); ax.grid(alpha=0.3, axis="y")
  plt.tight_layout(); plt.show()

for target_name in ("object_xy", "ee_xy"):
  _plot_grouped_r2(_by(results_rows, target=target_name),
                   f"{target_name}: probe R² by latent / source")

## Per-joint R²

In [ ]:
def _joint_matrix(rows, probe, metric="r2"):
  cols = [(s, k) for s in SOURCES for k in LATENT_KEYS]
  M = np.full((n_joints, len(cols)), np.nan)
  for j in range(n_joints):
    for c, (src, key) in enumerate(cols):
      hit = _by(rows, target=f"q_{j}", source=src, latent=key, probe=probe)
      if hit:
        M[j, c] = hit[0][metric]
  return M, [f"{s}/{k}" for s, k in cols]

fig, axes = plt.subplots(1, 2, figsize=(11, max(2.5, 0.35 * n_joints + 1.5)))
for ax, probe in zip(axes, ("linear", "mlp")):
  M, col_labels = _joint_matrix(results_rows, probe)
  im = ax.imshow(M, vmin=-0.1, vmax=1.0, aspect="auto", cmap="viridis")
  ax.set_xticks(np.arange(len(col_labels)))
  ax.set_xticklabels(col_labels, rotation=45, ha="right")
  ax.set_yticks(np.arange(n_joints))
  ax.set_yticklabels([f"q_{j}" for j in range(n_joints)])
  ax.set_title(f"per-joint R² — {probe} probe")
  for j in range(n_joints):
    for c in range(len(col_labels)):
      ax.text(c, j, f"{M[j, c]:.2f}", ha="center", va="center",
              color="white" if M[j, c] < 0.5 else "black", fontsize=7)
fig.colorbar(im, ax=axes, shrink=0.8, label="R²")
plt.show()

## Prior rollout: probe R² vs step

Best prior probe (MLP on `[h, s]`) re-evaluated per step. Tracks how fast
decodable information about each target decays under open-loop rollout.

In [ ]:
def _r2_by_step(target_key, col, eps_tr, eps_te):
  X_tr, _    = _pool(eps_tr, "hs", "prior")
  X_te, t_te = _pool(eps_te, "hs", "prior")
  y_tr = _pool_target(eps_tr, target_key, col=col)
  y_te = _pool_target(eps_te, target_key, col=col)
  _, pred_te, _ = fit_mlp_probe(X_tr, y_tr, X_te, y_te,
                                hidden=mlp_hidden, epochs=mlp_epochs,
                                lr=mlp_lr, batch=mlp_batch)
  T_max = int(t_te.max()) + 1
  out = np.full(T_max, np.nan)
  for t in range(T_max):
    mask = t_te == t
    if mask.sum() >= 2:
      out[t] = r2_score(y_te[mask], pred_te[mask])
  return out

targets_for_curve = [("object_xy", "object_xy", None),
                     ("ee_xy",     "ee_xy",     None),
                     ("q_0",       "joints",    0)]
curves = {name: _r2_by_step(key, col, eps_tr, eps_te)
          for name, key, col in targets_for_curve}

fig, ax = plt.subplots(figsize=(9, 4))
for (name, _, _), c in zip(targets_for_curve, ("C0", "C1", "C2")):
  curve = curves[name]
  ax.plot(np.arange(curve.shape[0]), curve, label=name, color=c, linewidth=2)
ax.axvline(burn_in - 0.5, color="k", linestyle=":", alpha=0.5, label="burn-in")
ax.set_xlabel("RSSM step index"); ax.set_ylabel("R² (prior / hs / mlp probe)")
ax.set_title("Prior-rollout probe R² vs step")
ax.set_ylim(min(-0.05, ax.get_ylim()[0]), 1.05)
ax.axhline(0, color="k", linewidth=0.5, alpha=0.4)
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## Summary

In [ ]:
def _aggregate(rows, target):
  return [{"source": r["source"], "latent": r["latent"], "probe": r["probe"],
           "r2": round(r["r2"], 4), "rmse": round(r["rmse"], 5)}
          for r in _by(rows, target=target)]

summary = {
  "checkpoint":       str(checkpoint_path),
  "data":             str(data_path),
  "obs_mode":         obs_mode,
  "act_mode":         ckpt.env.act_mode,
  "n_train_episodes": len(train_eps),
  "n_test_episodes":  len(test_eps),
  "burn_in":          burn_in,
  "horizon":          horizon,
  "object_xy":        _aggregate(results_rows, "object_xy"),
  "ee_xy":            _aggregate(results_rows, "ee_xy"),
  "joints_mean_r2": {
    f"{src}/{key}/{probe}": float(np.mean([
      r["r2"] for r in results_rows
      if r["source"] == src and r["latent"] == key and r["probe"] == probe
      and r["target"].startswith("q_")
    ])) for src in SOURCES for key in LATENT_KEYS for probe in PROBES
  },
}
print(json.dumps(summary, indent=2, default=str))